# kumo-tabular on a local NIM

One table. You have rows whose answer is known and rows whose answer you want,
and both go in one request.

This needs only the base install, `pip install structured-data-models-client`,
and a `kumo-tabular` NIM you point it at. The relational model is a separate
notebook, [`relational.ipynb`](relational.ipynb).

In [1]:
import os

import pandas as pd
from sdm_client import KumoTabular, Request, SDMClient

# Point this at your own deployment, or set the variable instead.
TABULAR_URL = os.environ.get('SDM_TABULAR_URL', 'http://localhost:8000')

## One table

You have rows where the answer is known, and rows you want scored. The rows
being scored need not appear in the first half, which is the point of the
model.


In [2]:
labelled = pd.DataFrame(
    {
        'age': [31.0, 44.5, 28.0, 51.0, 39.0, 47.0, 25.0, 58.0],
        'spend': [120.0, 430.0, 90.0, 510.0, 260.0, 480.0, 70.0, 620.0],
        'churn': [0, 1, 0, 1, 0, 1, 0, 1],
    }
)
to_score = pd.DataFrame({'age': [33.0, 47.0], 'spend': [150.0, 480.0]})

request = Request(
    context=labelled,
    predict=to_score,
    target='churn',
    task='classification',
)
request

Request(classification, target='churn', 8 context rows, 2 to score, single-table)

In [3]:
with SDMClient(url=TABULAR_URL) as client:
    print('the endpoint serves:', client.models())
    result = client.predict(request, model=KumoTabular())

result.to_pandas()

the endpoint serves: ['kumo-tabular']


,row_index,prediction,probability_0,probability_1
0,0,0,0.980807,0.019192
1,1,1,0.003052,0.996948


Join on the key the result carries, or on `row_index`. **Never on row
order**. The endpoint is not required to answer in the order you asked, and a
request split across several calls certainly will not.


In [4]:
answers = result.to_pandas()
to_score.reset_index(names='row_index').merge(answers, on='row_index')

,row_index,age,spend,prediction,probability_0,probability_1
0,0,33.0,150.0,0,0.980807,0.019192
1,1,47.0,480.0,1,0.003052,0.996948


## The same question, asked in PQL

`Request` names the two halves directly. The Predictive Query Language says
what to predict and which entity it is about. The rows to score are given to
`materialize`, and `predict=` takes the very same rows the direct request used,
so this asks exactly the question above and should give exactly the answers
above.

A query names its entity by a key, which a bare `Request` does not need, so the
two frames get one here.

In [5]:
from sdm_client import PredictiveQuery, Schema

keyed = labelled.assign(customer_id=range(len(labelled)))
keyed_to_score = to_score.assign(customer_id=[101, 102])

schema = Schema.from_data(
    {'customers': keyed}, primary_keys={'customers': 'customer_id'}
)
query = PredictiveQuery(
    'PREDICT customers.churn FOR EACH customers.customer_id', schema
)
request = query.materialize(
    predict=keyed_to_score, task='classification', model=KumoTabular()
)

with SDMClient(url=TABULAR_URL) as client:
    print(client.predict(request, model=KumoTabular()).to_pandas())

   row_index   id  prediction  probability_0  probability_1
0          0  101           0       0.980807       0.019192
1          1  102           1       0.003052       0.996948


The same two predictions, now keyed by `customer_id` rather than by position.

`materialize` needs to be told which rows are the question: `predict=` for rows
you hold, as here, or `indices=` to name rows already in the table. Supplying
neither is refused.

`task` is stated here for the same reason `Request` requires it. A query
would otherwise infer it, and `churn` holds two whole numbers, which could be
labels or quantities; leaving it out returns the same answer and warns that it
guessed. Pass `task='regression'` to ask the other question.

Almost every clause the relational language has is refused on one table: no
aggregations, no time windows, no `RANK`, no `ASSUMING`. Each refusal says
so.